# Ensemble: Chronos + a second model

Two jobs, in order.

**1. A diagnostic.** The GNN track rests on one assumption: that a station's
*neighbours* carry information its own history and local weather do not. That is
testable without building a graph - hand the neighbouring stations' series to
Chronos-2 as covariates and see whether the error moves. If it does, the spatial
premise is confirmed and a graph model is worth the effort. If it does not, that
is worth knowing **before** anyone rebuilds one.

**2. The blend.** Combine the Chronos forecast with a second model's, with the
weight fitted per pollutant and per heating regime.

## The rule that makes this honest

The blend weight is **fitted on 2023 windows and never on 2024**. A weight fitted
on the evaluation year is leakage, and it is the easy mistake here: it would make
any ensemble look good. So this notebook enumerates a *tuning* window set from
2023 using the identical eligibility rule as the frozen 2024 set, runs Chronos
over it, and fits there.

That is also why the blend is deliberately **one scalar per cell** - 6
pollutants x 2 regimes at most. Fitting anything richer (per station, per
horizon hour) on a single year of tuning data will overfit, and you will not be
able to tell from the numbers that it did.

## Why heating / non-heating

Chronos is weakest in the inversion months (autumn 0.895, winter 0.773; spring
0.657). A basin-wide inversion is a *shared* event across stations - exactly what
a spatial model can see and a univariate one cannot. So the two models' relative
strength should flip between regimes, and the fitted weights are themselves a
result: if the Chronos weight drops in the heating season, that is direct
evidence the second model contributes where the first struggles.

Heating season here is **October to March**, matching the earlier Sarajevo work.

## Running this

Needs a **T4 GPU**. Part 1 is a few minutes. Part 2 is the expensive one - it
runs Chronos over both the 2024 evaluation windows and a 2023 tuning sample, and
saves both to disk so the blend can be refitted later without re-running the
model.

If no partner predictions are available yet, Part 3 falls back to a **stand-in**
(the 7-day diurnal climatology). That is not a placeholder that fakes numbers -
it is a real, genuinely decorrelated weak model, so the harness produces a real
result and you can see exactly how much a weak partner is worth before the GNN
arrives.


In [8]:
# ---- bootstrap: shared artifacts + the dataset --------------------------
SHARED_FOLDER_ID = "1ivElb2AifDNvIg47Ii6dV_XP9aO14WSE"
CLEAN_FILE_ID    = "1QyfzghimgyRhQ735k42JetDEoK5kfGZJ"

SHARED = "shared"
CLEAN  = "bih_hourly_clean.csv"

import os, sys, json, subprocess
import numpy as np
import pandas as pd

POLLUTANTS = ["pm10", "pm25", "so2", "no2", "o3", "co"]
MET        = ["temperature", "wind_speed", "wind_u", "wind_v"]


def _gdown():
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    return gdown


if not os.path.exists(os.path.join(SHARED, "bih_shared.py")):
    print("fetching the shared artifacts")
    _gdown().download_folder(id=SHARED_FOLDER_ID, output=SHARED, quiet=True)

if not os.path.exists(CLEAN) or os.path.getsize(CLEAN) < 1e8:
    print("fetching bih_hourly_clean.csv (~167 MB)")
    _gdown().download(id=CLEAN_FILE_ID, output=CLEAN, quiet=False)

_hdr = pd.read_csv(CLEAN, nrows=0).columns.tolist()
_need = ["datetime", "station"] + POLLUTANTS + [f"{p}_filled" for p in POLLUTANTS] + MET
_missing = [c for c in _need if c not in _hdr]
if _missing:
    raise SystemExit(
        f"\n{CLEAN} is not the hourly clean export - missing {_missing}.\n"
        f"Delete the downloaded file before retrying, or the stale copy is reused."
    )

sys.path.append(SHARED)
import bih_shared as bs

SPLIT   = bs.load_split(SHARED)
GROUPS  = bs.load_groups(SHARED)
# Results live on Drive when it can be mounted, so a Colab disconnect does not
# take the cached predictions with it. Set DRIVE = False to keep them local.
DRIVE = True
RESULTS = "results_ensemble"
if DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS = "/content/drive/MyDrive/results_ensemble"
    except Exception as e:
        print("no Drive ({}), results stay on the local disk".format(e))
os.makedirs(RESULTS, exist_ok=True)
print("results ->", RESULTS)

try:
    import chronos
    from chronos import Chronos2Pipeline   # noqa: F401
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "chronos-forecasting>=2.3.0"], check=True)
    import chronos

import torch
from chronos import BaseChronosPipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu":
    print("\nNo GPU - Part 2 will take hours. Runtime > Change runtime type > T4 GPU.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
results -> /content/drive/MyDrive/results_ensemble
torch 2.11.0+cu128 | device: cuda


In [9]:
# ---- configuration ------------------------------------------------------
MODEL     = "amazon/chronos-2"
CONTEXT   = 512
HORIZON   = SPLIT["horizon_hours"]
QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
MEDIAN    = QUANTILES.index(0.5)
BATCH     = 64 if DEVICE == "cuda" else 16

KNOWN_FUTURE = ["temperature", "wind_speed"]   # observed - see the caveat below
PAST_ONLY    = ["wind_u", "wind_v"]

# Heating season, matching the earlier Sarajevo work.
HEATING_MONTHS = (10, 11, 12, 1, 2, 3)

# Part 1
DIAG_N        = 400      # Sarajevo windows for the spatial diagnostic
DIAG_POLS     = ["pm10", "no2", "pm25"]
N_NEIGHBOURS  = 3

# Part 2
EVAL_N  = None    # None = all 31,688 frozen 2024 windows
TUNE_N  = 6000    # 2023 windows to fit blend weights on; 12 scalars need no more

# Part 3
PARTNER_CSV = None   # path to the partner model's predictions; None = stand-in

# NOTE, carried from the Chronos notebook: the weather covariates are OBSERVED
# over the horizon, which is not knowable at forecast time. A deployment would
# feed a numerical weather forecast. Every number below inherits that caveat.
print("heating months:", HEATING_MONTHS, "| context:", CONTEXT, "| horizon:", HORIZON)

heating months: (10, 11, 12, 1, 2, 3) | context: 512 | horizon: 24


## Load the panel, the weather, and the station geometry

In [10]:
panel = bs.load_panel(CLEAN, SHARED)
HOURS = next(iter(panel.values())).index
POS   = {t: i for i, t in enumerate(HOURS)}


def load_met(clean_csv, index):
    """{station: (n_hours, n_met)} on the panel's gap-free hourly index."""
    df = pd.read_csv(clean_csv, usecols=["station", "datetime"] + MET,
                     parse_dates=["datetime"])
    return {st: (g.drop_duplicates("datetime").set_index("datetime")
                  .reindex(index)[MET].to_numpy("float32"))
            for st, g in df.groupby("station", sort=True)}


met    = load_met(CLEAN, HOURS)
coords = pd.read_csv(os.path.join(SHARED, "station_coords.csv"))
dist   = pd.read_csv(os.path.join(SHARED, "station_distance_km.csv"), index_col=0)
wins   = bs.load_windows(SHARED)


def season_of(ts):
    m = ts.month
    return ("winter" if m in (12, 1, 2) else "spring" if m in (3, 4, 5)
            else "summer" if m in (6, 7, 8) else "autumn")


def annotate(w):
    w = w.copy()
    w["season"] = w.origin.map(season_of)
    w["regime"] = np.where(w.origin.dt.month.isin(HEATING_MONTHS),
                           "heating", "non_heating")
    return w


wins = annotate(wins)
print(f"{len(panel)} series | {len(wins):,} frozen 2024 windows")
print(wins.groupby("regime").size().to_string())

/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

107 series | 31,688 frozen 2024 windows
regime
heating        15622
non_heating    16066


## Part 1 - does neighbour information exist?

Sarajevo is the strongest possible test: six stations inside ~10 km of the same
basin, sharing the same inversion, plus Ivan Sedlo 30 km out at 969 m as a
regional-background contrast. If cross-station signal cannot be found here, it is
not in this network.

Three configurations, on identical windows:

| config | what the model sees |
|---|---|
| `weather` | own history + temperature/wind (the current best Chronos setup) |
| `weather+nbr` | the above, plus the 3 nearest co-measuring stations' history |
| `nbr_only` | own history + neighbours, no weather - isolates the spatial term |

Neighbours are given as **past** covariates only. Their future is no more
knowable than the target's.

In [11]:
def nearest(station, pollutant, k=N_NEIGHBOURS):
    """The k nearest stations that also measure this pollutant. Not restricted
    to the same city - if the nearest co-measuring station is elsewhere, that is
    still the nearest one."""
    have = [s for s in dist.index if (s, pollutant) in panel and s != station]
    if not have or station not in dist.index:
        return []
    return list(dist.loc[station, have].sort_values().index[:k])


def make_inputs(chunk, weather=True, neighbours=False):
    """One dict per window; all share a schema, as Chronos-2 requires."""
    items = []
    for w in chunk.itertuples():
        i, lo = POS[w.origin], max(0, POS[w.origin] - CONTEXT)
        item = {"target": panel[(w.station, w.pollutant)].y.to_numpy("float32")[lo:i]}
        past, fut = {}, {}
        if weather:
            m = met.get(w.station)
            if m is None:
                m = np.full((len(HOURS), len(MET)), np.nan, "float32")
            for name in KNOWN_FUTURE + PAST_ONLY:
                j = MET.index(name)
                past[name] = m[lo:i, j]
                if name in KNOWN_FUTURE:
                    fut[name] = m[i:i + HORIZON, j]
        if neighbours:
            nb = nearest(w.station, w.pollutant)
            for k in range(N_NEIGHBOURS):
                # Absent neighbour -> an all-NaN series, which keeps the schema
                # identical across windows. Chronos-2 masks NaNs itself.
                if k < len(nb):
                    past[f"nbr{k+1}"] = panel[(nb[k], w.pollutant)].y.to_numpy("float32")[lo:i]
                else:
                    past[f"nbr{k+1}"] = np.full(i - lo, np.nan, "float32")
        if past:
            item["past_covariates"] = past
            if fut:
                item["future_covariates"] = fut
        items.append(item)
    return items


def forecast(w, pipeline, weather=True, neighbours=False, batch=BATCH, verbose=True,
             ckpt=None, ckpt_every=25):
    """Returns (n, 24, 9). Chronos-2's batch_size counts SERIES (target plus
    every covariate), so it is scaled by the schema width - getting that wrong
    is a silent 5x slowdown, not an error."""
    from time import time
    width = 1 + (len(KNOWN_FUTURE) + len(PAST_ONLY) if weather else 0) \
              + (N_NEIGHBOURS if neighbours else 0)

    # Checkpointing. `ckpt` names a partial file under RESULTS holding the rows
    # finished so far; a re-run picks up from row len(partial) instead of zero.
    # Rows are always appended in window order, so the prefix is unambiguous.
    part = os.path.join(RESULTS, f"chronos_{ckpt}.partial.npy") if ckpt else None
    out, start = [], 0
    if part and os.path.exists(part):
        prev = np.load(part)
        if len(prev) > len(w):
            print(f"  {os.path.basename(part)} has {len(prev):,} rows for a "
                  f"{len(w):,}-row set - ignoring it, delete it to be sure")
        else:
            out, start = [prev], len(prev)
            print(f"  resuming from {start:,}/{len(w):,} saved rows", flush=True)

    def _flush():
        if part and out:
            # write-then-rename, so a disconnect mid-save cannot leave a
            # truncated file. The temp name must end in .npy or np.save
            # appends it and os.replace looks for a file that never existed.
            tmp = part + ".tmp.npy"
            np.save(tmp, np.concatenate(out))
            os.replace(tmp, part)

    t0 = time()
    for i in range(start, len(w), batch):
        items = make_inputs(w.iloc[i:i + batch], weather, neighbours)
        q, _ = pipeline.predict_quantiles(items, prediction_length=HORIZON,
                                          quantile_levels=QUANTILES,
                                          batch_size=batch * width)
        out.append(np.stack([t.numpy().reshape(HORIZON, len(QUANTILES)) for t in q]))
        done = i + len(items)
        nb_done = (done - start + batch - 1) // batch
        if part and (nb_done % ckpt_every == 0 or done >= len(w)):
            _flush()
        if verbose and (nb_done % 25 == 0 or done >= len(w)):
            el, todo = time() - t0, len(w) - done
            rate = (done - start) / max(el, 1e-9)
            print(f"  {done:>6,}/{len(w):,}  {el:5.0f}s, ~{todo / max(rate, 1e-9):5.0f}s left",
                  flush=True)
    return np.clip(np.concatenate(out), 0, None)


def evaluate(preds_q, w, name):
    """Per-window MAE/RMSE/MASE/WQL. The MASE denominator always comes from the
    last 512 h, so these line up with baseline_windows.csv."""
    point, rows, wq = preds_q[:, :, MEDIAN], [], []
    for i, r in enumerate(w.itertuples()):
        y, mask = bs.target(panel, r.station, r.pollutant, r.origin)
        ctx = bs.context(panel, r.station, r.pollutant, r.origin, 512)
        rows.append({"model": name, **bs.score(point[i], y, mask, bs.mase_scale(ctx))})
        yt = np.where(mask, y, np.nan)[:, None]
        lv = np.array(QUANTILES)[None, :]
        d = yt - preds_q[i]
        pin = np.maximum(lv * d, (lv - 1) * d)
        den = np.nansum(np.abs(yt)) * len(QUANTILES)
        wq.append(np.nansum(pin) / den if den > 0 else np.nan)
    out = pd.concat([w.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    out["wql"] = wq
    return out


pipeline = BaseChronosPipeline.from_pretrained(MODEL, device_map=DEVICE,
                                               torch_dtype=torch.float32)
print("loaded", MODEL)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

loaded amazon/chronos-2


In [12]:
SARAJEVO = sorted(coords[coords.city == "Sarajevo"].station)
diag = wins[wins.station.isin(SARAJEVO) & wins.pollutant.isin(DIAG_POLS)]
diag = (diag.sample(min(DIAG_N, len(diag)), random_state=0)
            .sort_values(["station", "pollutant", "origin"]).reset_index(drop=True))
print(f"{len(diag)} Sarajevo windows, "
      f"{diag.groupby(['station','pollutant']).ngroups} series\n")

nb_km = [dist.loc[r.station, n] for r in diag.itertuples()
         for n in nearest(r.station, r.pollutant)]
print(f"median neighbour distance: {np.median(nb_km):.1f} km\n")

diag_rows = []
for name, kw in [("weather",     dict(weather=True,  neighbours=False)),
                 ("weather+nbr", dict(weather=True,  neighbours=True)),
                 ("nbr_only",    dict(weather=False, neighbours=True))]:
    diag_rows.append(evaluate(forecast(diag, pipeline, verbose=False, **kw), diag, name))
    print(f"{name} done", flush=True)

dg = pd.concat(diag_rows, ignore_index=True)
dg.to_csv(os.path.join(RESULTS, "spatial_diagnostic.csv"), index=False)

print("\nSPATIAL DIAGNOSTIC - MASE\n")
tab = dg.pivot_table(index="model", values="mase", aggfunc="mean").sort_values("mase")
tab["vs_weather_%"] = (1 - tab.mase / tab.loc["weather", "mase"]) * 100
print(tab.to_string(float_format=lambda v: f"{v:8.4f}"))
print("\nby regime\n")
print(dg.pivot_table(index="regime", columns="model", values="mase")
        .to_string(float_format=lambda v: f"{v:8.4f}"))
print("\nby pollutant\n")
print(dg.pivot_table(index="pollutant", columns="model", values="mase")
        .to_string(float_format=lambda v: f"{v:8.4f}"))

_gain = (1 - tab.loc["weather+nbr", "mase"] / tab.loc["weather", "mase"]) * 100
print(f"""

READ THIS AS:
  neighbours change MASE by {_gain:+.1f}% over the weather-only configuration.
  > +2%   spatial signal is real - a graph model has something to find.
  0 to +2%  marginal; the GNN would have to be very good to beat the blend cost.
  <= 0%   Chronos cannot use neighbours. That does NOT prove a GNN cannot -
          a graph model represents space explicitly, whereas Chronos sees the
          neighbours as unstructured extra series - but it removes the cheap
          evidence for it, and the GNN should then be justified on its own
          results rather than on the premise.""")

400 Sarajevo windows, 20 series

median neighbour distance: 7.5 km

weather done
weather+nbr done
nbr_only done

SPATIAL DIAGNOSTIC - MASE

                mase  vs_weather_%
model                             
weather+nbr   0.7768        2.1098
weather       0.7935        0.0000
nbr_only      0.8060       -1.5724

by regime

model        nbr_only  weather  weather+nbr
regime                                     
heating        0.9010   0.8691       0.8528
non_heating    0.7110   0.7179       0.7007

by pollutant

model      nbr_only  weather  weather+nbr
pollutant                                
no2          0.8133   0.7746       0.7669
pm10         0.7577   0.7746       0.7475
pm25         0.8686   0.8541       0.8385


READ THIS AS:
  neighbours change MASE by +2.1% over the weather-only configuration.
  > +2%   spatial signal is real - a graph model has something to find.
  0 to +2%  marginal; the GNN would have to be very good to beat the blend cost.
  <= 0%   Chronos cannot use nei

## Part 2 - Chronos predictions on both window sets

The blend needs Chronos forecasts on **two** sets:

- the frozen **2024** evaluation windows - what everything is finally scored on;
- a **2023 tuning** set - where the blend weights are fitted, so that no weight
  ever sees 2024.

There is no shared file of 2023 windows, so they are enumerated here with the
**identical** eligibility rule `shared_setup.ipynb` applies to 2024: at least 12
of the 24 target hours real and unfilled, and some history in the preceding
week. Same rule, different year - not a second convention.

Both sets are saved to disk. Refitting the blend later, or blending against a
different partner model, then costs nothing.

In [13]:
MIN_REAL_TARGET = SPLIT["masking"]["min_real_target_hours"]


def enumerate_windows(year):
    """The shared eligibility rule, applied to any year."""
    origins = HOURS[(HOURS.year == year) & (HOURS.hour == 0)]
    out = []
    for (st, p), s in panel.items():
        y, fl = s.y.to_numpy(), s.filled.to_numpy()
        real = ~np.isnan(y) & ~fl
        for o_ts in origins:
            o = POS[o_ts]
            if o + HORIZON > len(y) or o < 168:
                continue
            if int(real[o:o + HORIZON].sum()) < MIN_REAL_TARGET:
                continue
            if not np.isfinite(y[o - 168:o]).any():
                continue
            out.append({"station": st, "pollutant": p, "origin": o_ts})
    df = pd.DataFrame(out)
    meta = wins[["station", "city", "source"]].drop_duplicates("station")
    return annotate(df.merge(meta, on="station", how="left"))


def stratified(w, n, seed=0):
    """Proportional within pollutant x season, so a subsample keeps the mix."""
    if n is None or n >= len(w):
        return w.reset_index(drop=True)
    rng, frac, parts = np.random.RandomState(seed), n / len(w), []
    for _, g in w.groupby(["pollutant", "season"]):
        k = max(1, int(round(len(g) * frac)))
        parts.append(g.iloc[rng.choice(len(g), k, replace=False)])
    return (pd.concat(parts).sort_values(["station", "pollutant", "origin"])
              .reset_index(drop=True))


tune_all = enumerate_windows(2023)
tune = stratified(tune_all, TUNE_N, seed=1)
evalw = stratified(wins, EVAL_N, seed=0)
print(f"2023 tuning windows: {len(tune_all):,} eligible -> {len(tune):,} sampled")
print(f"2024 eval windows  : {len(evalw):,}")
print("\ntuning set by regime:")
print(tune.groupby(["pollutant", "regime"]).size().unstack(fill_value=0).to_string())

2023 tuning windows: 30,061 eligible -> 5,999 sampled
2024 eval windows  : 31,688

tuning set by regime:
regime     heating  non_heating
pollutant                      
co             410          396
no2            631          666
o3             401          419
pm10           566          555
pm25           378          391
so2            556          630


In [14]:
def run_and_save(w, tag):
    """Forecast and cache. Re-running the notebook reuses the saved file."""
    f = os.path.join(RESULTS, f"chronos_{tag}.npy")
    kf = os.path.join(RESULTS, f"chronos_{tag}_windows.csv")
    if os.path.exists(f):
        print(f"{tag}: reusing cached predictions")
        return np.load(f)
    print(f"{tag}: forecasting {len(w):,} windows")
    # The window list is written FIRST: a partial .npy is only interpretable
    # against the exact window ordering it was produced from.
    w.to_csv(kf, index=False)
    pq = forecast(w, pipeline, weather=True, neighbours=False, ckpt=tag)
    np.save(f, pq)
    part = os.path.join(RESULTS, f"chronos_{tag}.partial.npy")
    if os.path.exists(part):
        os.remove(part)
    return pq


pq_tune = run_and_save(tune,  "tune2023")
pq_eval = run_and_save(evalw, "eval2024")

res_tune = evaluate(pq_tune, tune,  "chronos2")
res_eval = evaluate(pq_eval, evalw, "chronos2")
print(f"\nchronos2 MASE - 2023 tuning {res_tune.mase.mean():.4f} "
      f"| 2024 eval {res_eval.mase.mean():.4f}")

tune2023: forecasting 5,999 windows
   1,600/5,999     31s, ~   84s left
   3,200/5,999     62s, ~   54s left
   4,800/5,999     92s, ~   23s left
   5,999/5,999    115s, ~    0s left
eval2024: forecasting 31,688 windows
   1,600/31,688     31s, ~  581s left
   3,200/31,688     62s, ~  550s left
   4,800/31,688     93s, ~  518s left
   6,400/31,688    123s, ~  486s left
   8,000/31,688    154s, ~  455s left
   9,600/31,688    185s, ~  425s left
  11,200/31,688    215s, ~  394s left
  12,800/31,688    246s, ~  363s left
  14,400/31,688    277s, ~  332s left
  16,000/31,688    307s, ~  301s left
  17,600/31,688    338s, ~  270s left
  19,200/31,688    369s, ~  240s left
  20,800/31,688    399s, ~  209s left
  22,400/31,688    430s, ~  178s left
  24,000/31,688    461s, ~  148s left
  25,600/31,688    491s, ~  117s left
  27,200/31,688    522s, ~   86s left
  28,800/31,688    553s, ~   55s left
  30,400/31,688    583s, ~   25s left
  31,688/31,688    608s, ~    0s left

chronos2 MASE - 20

## Part 3 - the blend

```
pred = w * chronos + (1 - w) * partner
```

with `w` fitted **per (pollutant, regime)** on the 2023 set by a grid search that
minimises mean MASE, then applied unchanged to 2024. `w` is clipped to [0, 1]: a
negative weight, or one above 1, is extrapolation that a single year of tuning
data cannot support.

Cells with too few tuning windows fall back to `w = 1` (Chronos alone) rather
than fitting a weight on noise. That fallback is reported, not silent.

**If no partner predictions exist yet**, a stand-in is used: the 7-day diurnal
climatology - the same-hour mean over the past week. It is a real model, it is
genuinely decorrelated from Chronos (pure local climatology, no level
forecasting), and it is weak (MASE ~1.07). So the result below is a real
measurement of what a weak, decorrelated partner is worth - a lower bound on
what the GNN would need to beat to be worth including.

In [15]:
def diurnal_standin(w, days=7):
    """Same-hour mean over the past week - the stand-in partner."""
    out = np.full((len(w), HORIZON), np.nan)
    for i, r in enumerate(w.itertuples()):
        y = panel[(r.station, r.pollutant)].y.to_numpy()
        j = POS[r.origin]
        c = y[j - days * 24:j].reshape(days, 24)
        with np.errstate(invalid="ignore"):
            m = np.nanmean(c, axis=0)
        fb = np.nanmean(y[max(0, j - 512):j])
        out[i] = np.where(np.isnan(m), fb, m)
    return out


def load_partner(path, w):
    """Read a partner model's predictions and align them to `w`, row for row.

    Accepts either the long format (one row per horizon hour, columns
    station / pollutant / origin / h / pred) or the wide one (h01..h24).
    Missing windows come back as NaN and are dropped from the comparison -
    never silently filled, which would let a partial partner look complete.
    """
    df = pd.read_csv(path, parse_dates=["origin"])
    if "h" in df.columns and "pred" in df.columns:
        df = df.pivot_table(index=["station", "pollutant", "origin"],
                            columns="h", values="pred")
        df = df.reindex(columns=range(1, HORIZON + 1))
    else:
        cols = [f"h{h:02d}" for h in range(1, HORIZON + 1)]
        miss = [c for c in cols if c not in df.columns]
        if miss:
            raise SystemExit(f"partner file is missing columns: {miss}")
        df = df.set_index(["station", "pollutant", "origin"])[cols]
    idx = pd.MultiIndex.from_frame(w[["station", "pollutant", "origin"]])
    got = df.reindex(idx)
    have = got.notna().all(axis=1).to_numpy()
    print(f"partner: {have.sum():,}/{len(w):,} windows matched "
          f"({have.mean():.1%})")
    return got.to_numpy(float), have


if PARTNER_CSV:
    partner_tune, ok_tune = load_partner(PARTNER_CSV, tune)
    partner_eval, ok_eval = load_partner(PARTNER_CSV, evalw)
    PARTNER_NAME = "partner"
else:
    print("no PARTNER_CSV set - using the 7-day diurnal climatology stand-in\n")
    partner_tune = diurnal_standin(tune)
    partner_eval = diurnal_standin(evalw)
    ok_tune = np.ones(len(tune), bool)
    ok_eval = np.ones(len(evalw), bool)
    PARTNER_NAME = "diurnal7_standin"

no PARTNER_CSV set - using the 7-day diurnal climatology stand-in



/tmp/ipykernel_1467/2231332537.py:9: RuntimeWarning: Mean of empty slice
  m = np.nanmean(c, axis=0)


In [16]:
MIN_CELL = 100          # tuning windows required before a weight is fitted
GRID = np.round(np.arange(0, 1.0001, 0.02), 3)


def masked_mase(pred, w, rows):
    """Mean MASE of `pred` over the given rows, using the shared mask."""
    out = []
    for i in np.flatnonzero(rows):
        r = w.iloc[i]
        y, mask = bs.target(panel, r.station, r.pollutant, r.origin)
        ctx = bs.context(panel, r.station, r.pollutant, r.origin, 512)
        s = bs.mase_scale(ctx)
        if not (np.isfinite(s) and s > 0) or not mask.any():
            continue
        out.append(np.nanmean(np.abs(np.where(mask, pred[i] - y, np.nan))) / s)
    return float(np.mean(out)) if out else np.nan


def fit_weights(chronos_pt, partner_pt, w, ok):
    """One scalar per (pollutant, regime), fitted on 2023 only."""
    rows = []
    for (pol, reg), g in w.groupby(["pollutant", "regime"]):
        sel = np.zeros(len(w), bool)
        sel[g.index.to_numpy()] = True
        sel &= ok
        n = int(sel.sum())
        if n < MIN_CELL:
            rows.append({"pollutant": pol, "regime": reg, "n_tune": n,
                         "w_chronos": 1.0, "fitted": False,
                         "mase_chronos": np.nan, "mase_blend": np.nan})
            continue
        scores = [masked_mase(a * chronos_pt + (1 - a) * partner_pt, w, sel) for a in GRID]
        best = int(np.nanargmin(scores))
        rows.append({"pollutant": pol, "regime": reg, "n_tune": n,
                     "w_chronos": float(GRID[best]), "fitted": True,
                     "mase_chronos": scores[-1], "mase_blend": scores[best]})
    return pd.DataFrame(rows)


W = fit_weights(pq_tune[:, :, MEDIAN], partner_tune, tune, ok_tune)
W["tune_gain_%"] = (1 - W.mase_blend / W.mase_chronos) * 100
W.to_csv(os.path.join(RESULTS, "blend_weights.csv"), index=False)

print("BLEND WEIGHTS - fitted on 2023 ONLY\n")
print(W.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print("""
w_chronos = 1.00 means the partner earned no weight in that cell.
A LOWER w in the heating rows is the result to look for: it means the partner
contributes where Chronos is weakest.""")
if not W.fitted.all():
    print(f"\n{(~W.fitted).sum()} cell(s) had under {MIN_CELL} tuning windows "
          f"and fall back to Chronos alone.")

BLEND WEIGHTS - fitted on 2023 ONLY

pollutant      regime  n_tune  w_chronos  fitted  mase_chronos  mase_blend  tune_gain_%
       co     heating     410     1.0000    True        0.6713      0.6713       0.0000
       co non_heating     396     1.0000    True        0.6194      0.6194       0.0000
      no2     heating     631     1.0000    True        0.6794      0.6794       0.0000
      no2 non_heating     666     0.9400    True        0.7356      0.7350       0.0878
       o3     heating     401     1.0000    True        0.6232      0.6232       0.0000
       o3 non_heating     419     0.9800    True        0.6815      0.6812       0.0449
     pm10     heating     566     0.9800    True        0.7156      0.7155       0.0165
     pm10 non_heating     555     1.0000    True        0.7048      0.7048       0.0000
     pm25     heating     378     1.0000    True        1.1014      1.1014       0.0000
     pm25 non_heating     391     0.9600    True        0.6780      0.6770       0.

In [17]:
wmap = {(r.pollutant, r.regime): r.w_chronos for r in W.itertuples()}
a = evalw.set_index(["pollutant", "regime"]).index.map(wmap).to_numpy(float)[:, None]

blend_point = a * pq_eval[:, :, MEDIAN] + (1 - a) * partner_eval
# Blend the whole distribution too, so WQL stays meaningful: the partner is a
# point forecast, so its "quantiles" are its point value repeated - which widens
# nothing and simply shifts the Chronos spread.
blend_q = a[:, :, None] * pq_eval + (1 - a[:, :, None]) * partner_eval[:, :, None]

rows = [res_eval]
rows.append(evaluate(np.repeat(partner_eval[:, :, None], len(QUANTILES), 2),
                     evalw, PARTNER_NAME))
rows.append(evaluate(blend_q, evalw, "ensemble"))

base = pd.read_csv(os.path.join(SHARED, "baseline_windows.csv"), parse_dates=["origin"])
key = ["station", "pollutant", "origin"]
base = base.merge(evalw[key].assign(_k=1), on=key, how="inner").drop(columns="_k")
base = annotate(base[base.model != "ratio_pm25"])   # pm25-only; not comparable

allm = pd.concat(rows + [base], ignore_index=True)
allm = allm[np.isin(allm.index, allm.index)]
allm.to_csv(os.path.join(RESULTS, "ensemble_window_metrics.csv"), index=False)

print("HEADLINE - 2024 frozen windows\n")
h = (allm.groupby("model").agg(windows=("mase", "size"), mase=("mase", "mean"),
                               mae=("mae", "mean"), wql=("wql", "mean"))
         .sort_values("mase"))
h["vs_t24_%"] = (1 - h.mase / h.loc["persistence_t24", "mase"]) * 100
print(h.to_string(float_format=lambda v: f"{v:8.4f}", na_rep="   -"))

for by in ["pollutant", "regime", "season"]:
    print(f"\n\nMASE by {by}\n")
    print(allm.pivot_table(index=by, columns="model", values="mase")
             .to_string(float_format=lambda v: f"{v:7.4f}"))

_c, _e = h.loc["chronos2", "mase"], h.loc["ensemble", "mase"]
print(f"""

VERDICT
  chronos alone {_c:.4f} -> ensemble {_e:.4f}  ({(1-_e/_c)*100:+.2f}%)
  The blend is only worth reporting if it beats Chronos on the 2024 set with
  weights that were fitted on 2023. A gain under ~1% is within the noise of a
  single held-out year - say so rather than claiming the ensemble wins.""")

HEADLINE - 2024 frozen windows

                  windows     mase      mae      wql  vs_t24_%
model                                                         
ensemble            31688   0.7647   7.1360   0.1451   28.9703
chronos2            31688   0.7649   7.1378   0.1449   28.9479
diurnal7_standin    31688   1.0706  10.6561   0.3125    0.5525
persistence_t24     31688   1.0766  10.4397        -    0.0000
persistence_last    31688   1.2823  12.2444        -  -19.1157


MASE by pollutant

model      chronos2  diurnal7_standin  ensemble  persistence_last  persistence_t24
pollutant                                                                         
co           0.7414            1.0719    0.7414            1.3941           1.0525
no2          0.7596            0.9656    0.7590            1.5117           1.0400
o3           0.7009            1.0440    0.7004            1.4030           1.0501
pm10         0.7749            1.1414    0.7747            1.1615           1.0767
pm25    

## Part 4 - what the GNN and xLSTM tracks must save

The blend above is computable the moment a second model saves its predictions in
this format. Until then it runs against the stand-in.

**File:** one CSV, long format, one row per forecast hour.

| column | type | meaning |
|---|---|---|
| `station` | str | exactly as in `eval_windows.csv` (e.g. `Ilidza`) |
| `pollutant` | str | one of `pm10 pm25 so2 no2 o3 co` |
| `origin` | timestamp | the window origin, `YYYY-MM-DD 00:00:00` |
| `h` | int | horizon hour, **1 to 24** (1 = the first hour after the origin) |
| `pred` | float | the point forecast, in native units (CO mg/m³, rest µg/m³) |

A wide format with columns `h01`..`h24` instead of `h`/`pred` is also accepted.

**Four things that will silently break the blend if they are wrong:**

1. **Predict every window in `eval_windows.csv` you can, and also the 2023
   tuning windows.** Without 2023 predictions there is nothing to fit the weight
   on, and fitting it on 2024 is leakage. Part 2's `enumerate_windows(2023)` is
   the code that generates that list - use it, do not invent a second rule.
2. **Do not fill gaps.** Missing windows must be absent or NaN, never
   interpolated or zero-filled. Rows that are absent are dropped from the
   comparison; rows quietly filled with a plausible number are not.
3. **Native units, not normalised.** If the model trained on scaled or
   log-transformed targets, invert the transform before saving.
4. **`h = 1` is the first hour after midnight.** An off-by-one here shifts the
   whole forecast and shows up as a mysteriously bad blend rather than an error.

Then set `PARTNER_CSV` at the top of this notebook and re-run from Part 3. Part
2's cached Chronos predictions are reused, so refitting costs seconds.

In [18]:
h.to_csv(os.path.join(RESULTS, "ensemble_headline.csv"))
meta = {
    "chronos_model": MODEL, "context_hours": CONTEXT, "horizon_hours": HORIZON,
    "partner": PARTNER_NAME,
    "partner_is_standin": PARTNER_CSV is None,
    "heating_months": list(HEATING_MONTHS),
    "weights_fitted_on": "2023 windows enumerated with the shared eligibility rule",
    "weights_never_fitted_on": "2024 (the evaluation year)",
    "tune_windows": int(len(tune)), "eval_windows": int(len(evalw)),
    "mase_chronos": float(h.loc["chronos2", "mase"]),
    "mase_ensemble": float(h.loc["ensemble", "mase"]),
    "caveat": ("weather covariates use OBSERVED future values; a deployment "
               "would substitute a numerical weather forecast"),
}
with open(os.path.join(RESULTS, "ensemble_run.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("written to", os.path.abspath(RESULTS))
for f in sorted(os.listdir(RESULTS)):
    print("  ", f)

written to /content/drive/MyDrive/results_ensemble
   blend_weights.csv
   chronos_eval2024.npy
   chronos_eval2024_windows.csv
   chronos_tune2023.npy
   chronos_tune2023_windows.csv
   ensemble_headline.csv
   ensemble_run.json
   ensemble_window_metrics.csv
   spatial_diagnostic.csv
